In [ ]:
!pip install pandas openpyxl
!pip install requests
!pip install beautifulsoup4

In [ ]:
import os
from datetime import datetime

import pandas as pd

import requests
from bs4 import BeautifulSoup

URL_BASE = "https://www.adorocinema.com"
URL_INICIO = "https://www.adorocinema.com/filmes/criticas-filmes/?page="

CAMINHO_RAIZ = os.getcwd()

DATA_LOG = datetime.now().strftime("%Y_%m_%d")

In [ ]:
def buscar_detalhamento_critica(dados_critica):
    try:
        url_critica = dados_critica["URL da Crítica"]
        resposta = requests.get(url_critica)

        soup = BeautifulSoup(resposta.content)

        sessao_ficha_tecnica = soup.find(class_="meta-body")
        if sessao_ficha_tecnica:
            data_lancamento = sessao_ficha_tecnica.find(class_="date").get_text().strip()
            duracao = sessao_ficha_tecnica.find_all(class_="meta-body-item")[0].get_text().split("|")[1].replace("\n", "").replace("\t", "").replace("  ", " ").strip()
            generos = sessao_ficha_tecnica.find_all(class_="meta-body-item")[0].get_text().split("|")[2].replace("\n", "").replace("\t", "").replace("  ", " ").strip()
            direcao = sessao_ficha_tecnica.find_all(class_="meta-body-item")[1].get_text().split("|")[0].replace("\n", "").replace("\t", "").replace("  ", " ").split(":")[1].strip()
            roteiro = sessao_ficha_tecnica.find_all(class_="meta-body-item")[2].get_text().split("|")[0].replace("\n", "").replace("\t", "").replace("  ", " ").replace("Roteiro", "")
            elenco = sessao_ficha_tecnica.find_all(class_="meta-body-item")[3].get_text().split("|")[0].replace("\n", "").replace("\t", "").replace("  ", " ").split(":")[1].strip()
            titulo_original = sessao_ficha_tecnica.find_all(class_="meta-body-item")[4].get_text().split("|")[0].replace("\n", "").replace("\t", "").replace("  ", " ").replace("Título original ", "")

        sessao_sinopse = soup.find(id="synopsis-details")
        if sessao_sinopse:
            classificacao = sessao_sinopse.find(class_="certificate-text").get_text().strip()
            sinopse = sessao_sinopse.find(class_="content-txt").get_text().strip()

        sessao_critica = soup.find(class_="ovw-editorial-block")
        if sessao_critica:
            nota = sessao_critica.find(class_="note").get_text().strip()
            conceito = sessao_critica.find(class_="light").get_text().strip()
            
            titulo_sessao_critica = sessao_critica.find(class_="editorial-header-title")
            titulo = titulo_sessao_critica.find(class_="title").get_text().strip()
            resumo = titulo_sessao_critica.find("p").get_text().strip()
            autor = sessao_critica.find(class_="editorial-header-title").get_text().strip()
            autor = autor.split("por ")[1] if "por " in autor else ""

            texto = sessao_critica.find(class_="editorial-content").get_text().strip().replace("\n", "").replace("\t", "").replace("  ", " ")

        dados_critica["Data de Lançamento"] = data_lancamento
        dados_critica["Duração"] = duracao
        dados_critica["Gêneros"] = generos
        dados_critica["Direção"] = direcao
        dados_critica["Roteiro"] = roteiro
        dados_critica["Elenco"] = elenco
        dados_critica["Título Original"] = titulo_original
        dados_critica["Classificação"] = classificacao
        dados_critica["Sinopse"] = sinopse
        dados_critica["Nota"] = nota
        dados_critica["Conceito"] = conceito
        dados_critica["Título da Crítica"] = titulo
        dados_critica["Resumo da Crítica"] = resumo
        dados_critica["Autor da Crítica"] = autor
        dados_critica["Texto da Crítica"] = texto
    except Exception as e:
        print(f"Falha ao coletar detalhamento da crítica: {e}")

    return dados_critica

In [ ]:
contador = 1
dados_criticas = []

sessao = requests.Session()

while True:
    url_pagina = f"{URL_INICIO}{contador}"
    resposta = sessao.get(url_pagina)

    if (
        contador > 1
        and str(contador) not in resposta.url
    ):
        break

    print(f"Executando para a página {contador}: {resposta.url}")
    soup = BeautifulSoup(resposta.content)

    print(soup.title)
    
    criticas = soup.find_all(class_="hred")

    for critica in criticas:
        url_imagem = critica.find(class_="thumbnail-img")["src"]

        elemento_titulo = critica.find(class_="meta-title-link")
        url_critica = URL_BASE + elemento_titulo["href"]
        titulo = elemento_titulo.get_text().strip()

        print(f"Coletando dados da crítica do filme: {titulo}")

        nota = critica.find(class_="stareval-note").get_text().strip()
        sinopse_resumida = critica.find(class_="synopsis").get_text().strip()

        dados_critica = {
            "URL da Imagem": url_imagem,
            "URL da Crítica": url_critica,
            "Título do Filme": titulo,
            "Nota": nota,
            "Sinopse Resumida": sinopse_resumida
        }

        dados_critica = buscar_detalhamento_critica(dados_critica)
        # print(dados_critica)

        dados_criticas.append(dados_critica)

    # Remover/comentar o if e o break para coletar todos os dados
    if contador > 1:
        break

    contador += 1

In [ ]:
caminho_relatorio = os.path.join(CAMINHO_RAIZ, f"coleta_dados_criticas_{DATA_LOG}.csv")

pd.DataFrame(dados_criticas).to_csv(caminho_relatorio, index=False)

In [ ]:
pd.DataFrame(dados_criticas)